# Шаг 3 Оценка: базовая модель vs дообученная

Сравниваю TinyLlama-1.1B базовую и дообученную на маркетинговом датасете.
Метрика: BERTScore — он лучше чем ROUGE для творческих текстов,
потому что учитывает семантическую близость а не точное совпадение слов.

Дополнительно: ручной разбор 5 примеров где стало лучше, где хуже и почему.

In [1]:
!pip install -q bert-score transformers>=4.41.0 peft accelerate
!pip install -q torchao --upgrade
!pip install -q bert-score transformers>=4.41.0 peft accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 40.1 MB/s eta 0:00:00


## Шаг 1 Загрузка адаптера

Загружаем marketing-lora-adapter.zip который скачали после обучения.

In [2]:
from google.colab import files
uploaded = files.upload()  # загружаю marketing-lora-adapter.zip

Saving marketing-lora-adapter.zip to marketing-lora-adapter (1).zip


In [3]:
import zipfile
with zipfile.ZipFile('marketing-lora-adapter.zip', 'r') as z:
    z.extractall('./marketing-lora-adapter')
print('Адаптер распакован')

import os
print('Файлы:', os.listdir('./marketing-lora-adapter'))

Адаптер распакован
Файлы: ['tokenizer_config.json', 'special_tokens_map.json', 'README.md', 'tokenizer.model', 'tokenizer.json', 'adapter_config.json', 'adapter_model.safetensors']


## Шаг 2 Загрузка обеих моделей

In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

MODEL_NAME = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
ADAPTER_PATH = './marketing-lora-adapter'

print('Загружаем токенизатор...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

print('Загружаем базовую модель...')
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='auto',
)

print('Загружаем дообученную модель (base + adapter)...')
finetuned_model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
finetuned_model.eval()

print('Обе модели загружены!')

Загружаем токенизатор...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Загружаем базовую модель...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Загружаем дообученную модель (base + adapter)...
Обе модели загружены!


## Шаг 3 Функция генерации

In [5]:
def generate(prompt, model, max_new_tokens=150):
    formatted = (
        f"<|system|>You are a professional marketing copywriter. "
        f"Write clear, concise, and compelling marketing content.</s>"
        f"<|user|>{prompt}</s>"
        f"<|assistant|>"
    )
    inputs = tokenizer(formatted, return_tensors='pt').to('cuda')
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.split('<|assistant|>')[-1].strip()

## Шаг 4 15 тестовых промптов

In [6]:
test_prompts = [
    'Write a catchy Instagram caption for a new coffee brand targeting young professionals.',
    'Create a slogan for an eco-friendly water bottle.',
    'Write a subject line for a promotional email about a 50% sale.',
    'Write a product description for wireless noise-canceling headphones.',
    'Create a Facebook ad for a fitness app targeting busy parents.',
    'Write a tagline for a luxury skincare brand.',
    'Write a promotional email for a new restaurant opening.',
    'Create a Twitter post announcing a new product launch.',
    'Write a call-to-action for a landing page selling online courses.',
    'Create a marketing message for a back-to-school sale.',
    'Write a headline for a travel agency summer campaign.',
    'Create a slogan for a pet food brand focused on natural ingredients.',
    'Write an email subject line for a flash sale ending tonight.',
    'Create a product description for a smart home device.',
    'Write a social media post for a charity fundraising campaign.',
]

print(f'Тестовых примеров: {len(test_prompts)}')
print('Генерируем ответы базовой модели...')
base_responses = [generate(p, base_model) for p in test_prompts]
print('Генерируем ответы дообученной модели...')
finetuned_responses = [generate(p, finetuned_model) for p in test_prompts]
print('Готово!')

Тестовых примеров: 15
Генерируем ответы базовой модели...


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

Генерируем ответы дообученной модели...


[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hugging

Готово!


## Шаг 5 BERTScore

In [7]:
from bert_score import score

# Считаю BERTScore: сравнивала ответы дообученной модели с базовой
# как референс использовала базовую — смотрю насколько они расходятся по смыслу
P, R, F1 = score(
    finetuned_responses,
    base_responses,
    lang='en',
    verbose=True,
)

print(f'\nBERTScore результаты:')
print(f'Precision: {P.mean():.4f}')
print(f'Recall:    {R.mean():.4f}')
print(f'F1:        {F1.mean():.4f}')
print(f'\nПо примерам (F1):')
for i, (p, f) in enumerate(zip(test_prompts, F1)):
    print(f'[{i+1:2d}] {f:.4f} | {p[:60]}')

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 0.89 seconds, 16.95 sentences/sec

BERTScore результаты:
Precision: 0.8706
Recall:    0.8750
F1:        0.8727

По примерам (F1):
[ 1] 0.8475 | Write a catchy Instagram caption for a new coffee brand targ
[ 2] 0.9046 | Create a slogan for an eco-friendly water bottle.
[ 3] 0.8473 | Write a subject line for a promotional email about a 50% sal
[ 4] 0.8874 | Write a product description for wireless noise-canceling hea
[ 5] 0.8619 | Create a Facebook ad for a fitness app targeting busy parent
[ 6] 0.8743 | Write a tagline for a luxury skincare brand.
[ 7] 0.9027 | Write a promotional email for a new restaurant opening.
[ 8] 0.8745 | Create a Twitter post announcing a new product launch.
[ 9] 0.8802 | Write a call-to-action for a landing page selling online cou
[10] 0.9053 | Create a marketing message for a back-to-school sale.
[11] 0.8688 | Write a headline for a travel agency summer campaign.
[12] 0.8598 | Create a slogan for a pet food brand focused on natural ingr
[13] 0.8473 | 

## Шаг 6 Ручной разбор 5 примеров

In [8]:

# Смотрим на примеры с наибольшим расхождением между моделями
import numpy as np
indices = np.argsort(F1.numpy())[:5]  # 5 примеров с наименьшим BERTScore F1

print(' РУЧНОЙ РАЗБОР: примеры где модели расходятся больше всего\n')
for i in indices:
    print(f'ПРОМПТ: {test_prompts[i]}')
    print(f'\nБАЗОВАЯ МОДЕЛЬ:')
    print(base_responses[i])
    print(f'\nДООБУЧЕННАЯ МОДЕЛЬ:')
    print(finetuned_responses[i])
    print(f'\nBERTScore F1: {F1[i]:.4f}')
    print('=' * 70)

=== РУЧНОЙ РАЗБОР: примеры где модели расходятся больше всего ===

ПРОМПТ: Write a social media post for a charity fundraising campaign.

БАЗОВАЯ МОДЕЛЬ:
Dear friends,

Join us in supporting a great cause and help us raise money for a deserving charity.

Join us on [insert social media platform] as we participate in [insert social media challenge, such as #hashtag or #meme] to raise awareness and funds for this cause.

Help us make a difference in the lives of those in need, and together we can make a positive impact.

Thank you for coming along and supporting our cause. Together, we can make a difference.

#CharityFundraiser

Join us at [insert social media platform] to help us raise awareness and funds for [insert charity name].

ДООБУЧЕННАЯ МОДЕЛЬ:
Introducing @charity_name, a non-profit organization that is dedicated to supporting those in need! Help us #MakeA Difference and #SupportCharity by donating, volunteering, or spreading the word! #GivingTuesday #CharityFundraising #Spread

## Шаг 7 Итоговые выводы

Мои наблюдения после просмотра примеров:
Несмотря на результаты метрик (BERTScore ~0.84-0.85). модели ведут себя абсолютно по-разному. Моя дообученная модель лушчше:
Модель шикарно уловила формат полноценных постов для соцсетей.
* **Пример:** Промпт на пост для благотворительности.
* **Базовая:** Выдала какое-то сухое официальное письмо («Dear friends...») и прилепила один хештег для галочки. Соцсетями там и не пахнет.
* **Моя (QLoRA):** Отработала идеально. Нормальная структура, релевантные хештеги (типа `#GivingTuesday`), добавила плейсхолдеры для картинок и четкий призыв к действию (CTA). Вайб пойман на 100%.

Где базовая модель справилась лучше:
Короткие форматы (слоганы, темы писем). Здесь моя модель проигрывает.
* **Пример:** Промпт на генерацию короткого слогана для корма.
* **Базовая:** Честно выдала ровно одну строчку, как я и просила.
* **Моя (QLoRA):** Вообще проигнорировала ограничение формата. Вместо слогана вывалила мне огромный рекламный пост с эмодзи, который в итоге еще и оборвался на полуслове. С темами для email та же беда — сразу начинает писать само письмо.

Правда моя модель немного переобучилась потому что пишет только длинные email-рассылку или радостный промо-пост.

**Format collapse (потеря форматов):** Модель практически разучилась писать коротко. Что бы я у нее ни попросила, она пытается раздуть это в длинную email-рассылку или радостный промо-пост.
2. **Эмоциональный передоз:** Появился агрессивно-маркетинговый тон. Везде лезут клише («Introducing...», «Hurry up!»), обилие восклицательных знаков и спам эмодзи (🎉, 🙌, 🔥), которых у базовой модели не было.
3. **Обрывы генерации:** В нескольких примерах текст обрывается на полуслове. Скорее всего, модель тупо упирается в лимит `max_new_tokens`, либо на этапе подготовки датасета я криво настроила EOS-токен (End of Sequence), и она просто не понимает, когда ей нужно остановиться.

Что буду фиксить в следующей итерации:
* **Сбалансировать датасет:** Обязательно добавлю больше примеров с короткими ответами (темы писем, слоганы), чтобы вернуть модели вариативность и отучить писать полотнами.
* **Починить EOS-токены:** Проверю логику токенизации, чтобы модель научилась нормально завершать мысль, а не обрываться.

In [9]:
print('ИТОГ')
print('=' * 50)
print(f'Средний BERTScore F1: {F1.mean():.4f}')
print(f'Лучший пример F1:    {F1.max():.4f}')
print(f'Худший пример F1:    {F1.min():.4f}')
print()
print('Loss дообученной модели:')
print('  Epoch 1: train=1.248, val=0.971')
print('  Epoch 2: train=0.987, val=0.953')
print('  Epoch 3: train=0.992, val=0.952')
print()
print('Вывод: val loss стабилизировался — переобучения нет.')

ИТОГ
Средний BERTScore F1: 0.8727
Лучший пример F1:    0.9053
Худший пример F1:    0.8338

Loss дообученной модели:
  Epoch 1: train=1.248, val=0.971
  Epoch 2: train=0.987, val=0.953
  Epoch 3: train=0.992, val=0.952

Вывод: val loss стабилизировался — переобучения нет.
